# Foundation Model Fine-Tuning
**NB 23** | Ethiopian Highlands | DINOv3 SAT + Prithvi-EO-2.0 cookbook

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import pygeovision as pgv
from pygeovision.models import get_model
from pygeovision.inference.tiled import TiledInference

client = pgv.PyGeoVision()
print(client)
from pygeovision.models.foundation.dinov3 import DINOv3Backbone, get_transform, finetune_dinov3
from pygeovision.models.foundation.prithvi import PrithviTasks, finetune_prithvi


In [ ]:
BBOX       = (38.6, 8.9, 38.95, 9.15)
DATE_RANGE = ('2024-01-01', '2024-06-30')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 15,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32637', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")


In [ ]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 6,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


In [ ]:
PREPROCESSED = DATA_DIR / 'ethiopia_foundation_ready.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'foundation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [ ]:
# ── DINOv3 SAT feature extraction (ALWAYS use get_transform for correct stats) ──
backbone  = DINOv3Backbone('dinov3_vitl16_sat')
transform = get_transform('dinov3_vitl16_sat')   # SAT_MEAN/SAT_STD ← critical
if PREPROCESSED.exists():
    embeddings = backbone.extract_embeddings(str(PREPROCESSED))   # (1, 1024)
    features   = backbone.extract_features(str(PREPROCESSED))     # (H_p, W_p, 1024)
    print(f"Global embedding : {embeddings.shape}  dtype={embeddings.dtype}")
    print(f"Spatial features : {features.shape}")
    print(f"Pre-training     : SAT-493M (493M satellite image patches)")
    print(f"Embedding dim    : 1024 (ViT-L/16)")


In [ ]:
# ── DINOv3 Fine-tuning configuration (segmentation) ──────────────────────────
from pygeovision.training.data import GeoSegDataset
from torch.utils.data import DataLoader

dinov3_config = {
    'model_name'       : 'dinov3_vitl16_sat',
    'task'             : 'segmentation',
    'num_classes'      : 9,              # ESA WorldCover 9 classes
    'epochs'           : 50,
    'learning_rate'    : 1e-5,           # low LR critical for SAT pre-trained ViT
    'weight_decay'     : 0.05,
    'mixed_precision'  : True,           # BF16 mandatory for ViT-L (300M)
}
print("DINOv3 fine-tuning configuration:")
for k, v in dinov3_config.items():
    print(f"  {k:<20}: {v}")
print("\nNotes:")
print("  • learning_rate=1e-5 (NOT 6e-5 — SAT pre-trained models need lower LR)")
print("  • mixed_precision=True — ViT-L requires BF16 to fit in 40GB GPU")
print("  • get_transform() MUST be used — wrong stats = -10 to -20 mIoU silently")


In [ ]:
# ── Prithvi-EO-2.0 Fine-tuning configuration ─────────────────────────────────
prithvi_config = {
    'model_name'       : 'prithvi_eo_2_0',
    'task'             : 'land_cover',
    'num_classes'      : 9,
    'epochs'           : 50,
    'learning_rate'    : 5e-5,
    'batch_size'       : 4,              # memory-limited for 600M model
    'mixed_precision'  : True,
}
print("Prithvi-EO-2.0 fine-tuning configuration:")
for k, v in prithvi_config.items():
    print(f"  {k:<20}: {v}")
print("\nNotes:")
print("  • ALWAYS call map_bands(data, source='sentinel2') before inference")
print("  • HLS band order: Blue(B02) Green(B03) Red(B04) NIR(B08) SWIR1(B11) SWIR2(B12)")
print("  • normalise_hls() divides by HLS_SCALE_FACTOR=10000.0")
print("  • batch_size=4 (600M params needs 40GB GPU at batch_size=8)")


In [ ]:
# ── Comparison of fine-tuned foundation models ─────────────────────────────
print("\nFoundation Model Comparison (Ethiopia land cover task):")
print(f"{'Model':<28} {'Params':>8} {'Pre-training':<25} {'Typical mIoU'}")
print('-'*78)
models = [
    ('segformer-b2 (baseline)',  '27.5M', 'ImageNet',         '72-78'),
    ('dinov3_vitl16',            '300M',  'Web LVD-1689M',    '78-83'),
    ('dinov3_vitl16_sat',        '300M',  'SAT-493M ★',       '83-88'),
    ('prithvi_eo_2_0',           '600M',  'HLS Global 10yr ★','86-91'),
    ('dinov3_vit7b16_sat',       '6.7B',  'SAT-493M ★',       '88-93'),
]
for name, params, pre, miou in models:
    print(f"{name:<28} {params:>8} {pre:<25} {miou}")
print("\n★ = satellite-specific pre-training — highest accuracy for EO tasks")


In [ ]:
PREDICTION = DATA_DIR / 'foundation_finetuned_prediction.tif'

if ready and PREPROCESSED.exists():
    model = get_model('segformer-b2', num_classes=9, pretrained=True)
    inf   = TiledInference(model, chip_size=512, overlap=64, blend_mode='gaussian')
    res   = inf.infer(str(PREPROCESSED), str(PREDICTION))
    print(f"Inference: {res.get('n_chips')} chips  {res.get('duration_seconds',0):.1f}s  "
          f"{res.get('chips_per_second',0):.1f} chips/s")
else:
    print("Skipping inference — no preprocessed scene")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=50, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=0, min_area_m2=500.0, simplify_tolerance=0.5)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


**NB23 — Foundation Fine-tuning** complete.
- Study area: Ethiopian Highlands
- Sensor: Sentinel-2 6-band
- Model: DINOv3 SAT + Prithvi-EO-2.0
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG